<a href="https://colab.research.google.com/github/climatescience/Spatial-data-visualization-with-Python/blob/main/multi_panel_raster_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors  # Imported for setting up discrete boundaries

# 1. Gather files
file_list = sorted(glob.glob("/content/*.tif"))[:5]
if not file_list:
    file_list = sorted(glob.glob("/content/*.tif"))[:5]

season_names = ['Winter', 'Spring', 'Summer', 'Autumn']
panel_labels = ['b', 'c', 'd', 'e']

data_list = []
extent_list = []
spatial_means = []
vmin, vmax = float('inf'), float('-inf')

# 2. Extract data and global limits
for file in file_list:
    with rasterio.open(file) as src:
        data = src.read(1).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        spatial_means.append(np.nanmean(data))
        data_list.append(data)

        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        extent_list.append(extent)

        vmin = min(vmin, np.nanmin(data))
        vmax = max(vmax, np.nanmax(data))

annual_idx = 4
seasonal_indices = [0, 1, 2, 3]

# --- CRITICAL COLOR PALETTE TUNING FOR SMALL VARIATIONS ---
# 1. Create a stepped color classification system (10 clear intervals)
num_classes = 10
bounds_ticks = np.linspace(vmin, vmax, num_classes + 1)

# 2. Using 'jet' or 'Turbo' instead of inferno provides a wider spectral variation (Blue to Red)
# mcolors.BoundaryNorm forces matplotlib to drop seamless blending and show hard boundaries
base_cmap = plt.colormaps['turbo']
cmap = mcolors.ListedColormap(base_cmap(np.linspace(0, 1, num_classes)))
norm = mcolors.BoundaryNorm(bounds_ticks, cmap.N)

# 3. Create the Custom GridSpec Layout
fig = plt.figure(figsize=(14, 11))
gs = gridspec.GridSpec(3, 2, height_ratios=[1.2, 1, 1], hspace=0.45, wspace=0.2)

# --- Row 1: Elongated Annual Map ---
ax_annual = fig.add_subplot(gs[0, :])
im = ax_annual.imshow(data_list[annual_idx], cmap=cmap, norm=norm,
                       extent=extent_list[annual_idx], aspect='auto')
ax_annual.set_xlim(extent_list[annual_idx][0], extent_list[annual_idx][1])
ax_annual.set_ylim(extent_list[annual_idx][2], extent_list[annual_idx][3])
ax_annual.grid(True, which='major', linestyle=':', color='grey', alpha=0.35, linewidth=1)
ax_annual.set_axisbelow(False)

# Add this right after ax_annual.imshow(...)
ax_annual.text(0.015, 0.92, "(a)", transform=ax_annual.transAxes,
               fontsize=12, fontweight='bold', va='top', ha='left',
               bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
ax_annual.tick_params(labelsize=9)

# --- Rows 2 & 3: 2x2 Seasonal Grid ---
seasonal_axes = []
for idx, s_idx in enumerate(seasonal_indices):
    row = 1 + (idx // 2)
    col = idx % 2

    ax = fig.add_subplot(gs[row, col])
    seasonal_axes.append(ax)

    ax.imshow(data_list[s_idx], cmap=cmap, norm=norm,
              extent=extent_list[s_idx], aspect='auto')
    ax.set_xlim(extent_list[s_idx][0], extent_list[s_idx][1])
    ax.set_ylim(extent_list[s_idx][2], extent_list[s_idx][3])
    ax.grid(True, which='major', linestyle=':', color='grey', alpha=0.35, linewidth=1)
    ax.set_axisbelow(False)

    ax.text(0.03, 0.92, f"({panel_labels[idx]})", transform=ax.transAxes,
        fontsize=12, fontweight='bold', va='top', ha='left',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
ax.tick_params(labelsize=9)


# 4. Add the Shared Horizontal Colorbar at the Bottom
all_axes = [ax_annual] + seasonal_axes
cbar = fig.colorbar(im, ax=all_axes, orientation='horizontal',
                    shrink=0.6, aspect=40, pad=0.08, ticks=bounds_ticks)
cbar.set_label('Ozone (ppbv)', fontsize=12, labelpad=8)
cbar.ax.tick_params(labelsize=10)

# Format legend values nicely to match decimal step ranges
cbar.ax.set_xticklabels([f"{x:.1f}" for x in bounds_ticks])

plt.show()